1. The "best" AI model for forex trading depends heavily on what you're trying to do—predict price movements, analyze sentiment, automate execution, or manage risk. Here's a practical breakdown:
1. For Price Prediction / Technical Analysis
LSTM/GRU Networks: Still widely used for time-series forecasting of currency pairs. They handle sequential forex data well but require careful feature engineering.
Transformers (like Temporal Fusion Transformers): Increasingly popular for multivariate time-series, especially when combining technical indicators with macroeconomic data.
XGBoost/LightGBM: Often outperform deep learning for short-term directional predictions when fed with engineered features. Many prop shops use gradient boosting as a baseline.
2. For Sentiment & News Analysis
LLMs (GPT-4, Claude, open-source models like Llama 3): Excellent for parsing central bank statements, geopolitical news, and social media sentiment. Real-time sentiment shifts often precede volatility.
Fine-tuned BERT/RoBERTa: More cost-effective if you only need classification (bullish/bearish/neutral) on financial text.
3. For Execution & High-Frequency Trading
Reinforcement Learning (PPO, DQN): Used for optimizing order execution, slippage minimization, and market-making. Requires low-latency infrastructure.
Classical algorithms (VWAP, TWAP) with ML overlays: More common than pure AI for execution.
4. For Risk Management
Anomaly detection models (Isolation Forests, Autoencoders): Flag unusual market regimes or portfolio drawdowns.
Regime-switching models (HMM): Detect shifts between trending and ranging markets.
Key Reality Checks
No model consistently predicts forex: The market is highly efficient, noisy, and influenced by unpredictable macro events. Most successful "AI trading" uses models for risk management or execution efficiency, not directional alpha.
Data quality matters more than model architecture: Clean tick data, proper handling of look-ahead bias, and realistic backtesting are critical.
Overfitting is the biggest risk: Forex has low signal-to-noise. Complex models often fail out-of-sample.
Practical Recommendation
If you're starting out, a stacked approach works best: use LLMs for sentiment/news analysis, gradient boosting for short-term signal generation, and strict risk management rules. For retail traders, platforms like MetaTrader with Python ML integration, or cloud services (AWS SageMaker, Google Vertex AI) are common infrastructure choices.
What's your specific use case—automated trading, signal generation, or risk analysis? That would help narrow down the most suitable architecture.

XGBoost filter pipeline for forex. This covers feature engineering, model training with proper validation, and live inference integration.

Building a forex automation trading system involves several layers beyond just picking a model. Here's a practical roadmap:
Architecture Overview
plain
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Data Layer    │────▶│  Strategy Layer │────▶│ Execution Layer │
│  (Ingestion)    │     │  (Models/Logic) │     │  (Order Mgmt)   │
└─────────────────┘     └─────────────────┘     └─────────────────┘
         │                                               │
         └───────────────────────────────────────────────┘
                    Risk Management (across all layers)
1. Data Layer
Market Data: Real-time tick/quote data via brokers (REST/WebSocket APIs like OANDA, Interactive Brokers, cTrader)
Alternative Data: Economic calendars, news feeds, sentiment data
Storage: Time-series databases (InfluxDB, TimescaleDB) for efficient querying
2. Strategy Layer (Your AI/ML Component)
Table
Approach	Best For	Complexity
Rule-based + ML overlay	Beginners, explainability	Low-Medium
LSTM/GRU	Pattern recognition in price sequences	Medium
Reinforcement Learning	Dynamic position sizing, execution optimization	High
Ensemble (XGBoost + LSTM)	Robust signal generation	High
Practical tip: Start with a simple rule-based system (e.g., moving average crossover) and gradually add ML components. Most profitable systems are 80% risk management, 20% signal.
3. Execution Layer
Paper trading first: Test with simulated orders
Broker API integration: OANDA v20, Interactive Brokers API, MetaTrader 5 via ZeroMQ
Order types: Market, limit, stop-loss, trailing stops
Latency considerations: Co-located servers if doing high-frequency
4. Risk Management (Critical)
Position sizing: Kelly criterion, fixed fractional, or volatility-based
Stop-losses: Hard stops + dynamic trailing stops
Drawdown limits: Kill switch if daily/weekly loss exceeds threshold
Correlation checks: Avoid overexposure to correlated pairs (e.g., EUR/USD and GBP/USD)
5. Infrastructure & Monitoring
Deployment: Docker containers, cloud VPS (AWS, GCP, Azure)
Logging: All trades, model predictions, and system events
Monitoring: Uptime alerts, P&L dashboards, model drift detection
Fail-safes: What happens if your server crashes mid-trade?
Tech Stack Example
Table
Component	Options
Language	Python (most popular), C++ (for latency), MQL5 (MetaTrader)
Data	pandas, numpy, yfinance, alpha_vantage
ML	scikit-learn, TensorFlow/PyTorch, XGBoost, Stable Baselines3 (RL)
Execution	ccxt (unified exchange API), broker-specific SDKs
Orchestration	Apache Airflow, cron, or custom schedulers
Common Pitfalls
Look-ahead bias: Using future data in backtests (e.g., training on adjusted close)
Overfitting: Models that work on historical data but fail live
Slippage & spread: Real execution costs erode theoretical profits
Survivorship bias: Only testing on pairs/assets that still exist
Regime change: Models trained on bull markets fail in crashes
Getting Started (Minimal Viable System)
Pick one pair (e.g., EUR/USD) and one timeframe
Build a simple strategy (e.g., RSI + moving average)
Backtest with realistic spreads/slippage
Paper trade for 1-3 months
Deploy with tiny position sizes
Iterate and add complexity slowly
Would you like me to elaborate on any specific layer—such as backtesting frameworks, RL implementation, or broker API integration?

In [10]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple

class ForexFeatureEngineer:
    """Feature engineering for XGBoost forex filter."""
    
    def __init__(self, lookback_periods: List[int] = [5, 10, 20, 50]):
        self.lookbacks = lookback_periods
    
    def compute_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        df columns: ['open', 'high', 'low', 'close', 'volume']
        Returns DataFrame with engineered features.
        """
        df = df.copy()
        
        # Price-based features
        for period in self.lookbacks:
            # Moving averages
            df[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()
            df[f'sma_{period}'] = df['close'].rolling(window=period).mean()
            
            # Distance from moving averages
            df[f'close_dist_ema_{period}'] = (df['close'] - df[f'ema_{period}']) / df[f'ema_{period}']
            
            # Volatility
            df[f'atr_{period}'] = self._atr(df, period)
            df[f'volatility_{period}'] = df['close'].rolling(period).std() / df['close'].rolling(period).mean()
            
            # Momentum
            df[f'momentum_{period}'] = df['close'].pct_change(period)
            df[f'roc_{period}'] = (df['close'] - df['close'].shift(period)) / df['close'].shift(period)
        
        # Candlestick features
        df['body_size'] = abs(df['close'] - df['open']) / df['open']
        df['upper_wick'] = (df['high'] - df[['close', 'open']].max(axis=1)) / df['open']
        df['lower_wick'] = (df[['close', 'open']].min(axis=1) - df['low']) / df['open']
        df['range'] = (df['high'] - df['low']) / df['low']
        
        # Volume features (if available)
        if 'volume' in df.columns and df['volume'].notna().any():
            df['volume_sma_20'] = df['volume'].rolling(20).mean()
            df['volume_ratio'] = df['volume'] / df['volume_sma_20']
        
        # Cross-pair features (if you track multiple pairs)
        # df['eur_gbp_ratio'] = eur_close / gbp_close  # example
        
        # Time features
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        
        return df
    
    def _atr(self, df: pd.DataFrame, period: int) -> pd.Series:
        """Average True Range."""
        high_low = df['high'] - df['low']
        high_close = abs(df['high'] - df['close'].shift())
        low_close = abs(df['low'] - df['close'].shift())
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()
    
    def get_feature_columns(self, df: pd.DataFrame) -> List[str]:
        """Return feature column names (excluding raw price/volume/target)."""
        exclude = ['open', 'high', 'low', 'close', 'volume', 'target']
        return [c for c in df.columns if c not in exclude]

In [11]:
class LabelGenerator:
    """
    Generate labels for XGBoost training.
    
    Target: Will price move 1R in favor within N bars?
    R = risk amount (e.g., 1 ATR)
    """
    
    def __init__(self, risk_multiple: float = 1.0,  # 1R
                 reward_multiple: float = 2.0,      # 2:1 R/R
                 max_bars: int = 20,                # Look ahead window
                 atr_period: int = 14):
        self.risk_multiple = risk_multiple
        self.reward_multiple = reward_multiple
        self.max_bars = max_bars
        self.atr_period = atr_period
    
    def generate_labels(self, df: pd.DataFrame, direction: str = 'long') -> pd.Series:
        """
        For each bar, determine if a trade taken at that bar's close
        would hit reward before hitting risk within max_bars.
        
        Returns: 1 (win), 0 (loss or timeout), -1 (skip/uncertain)
        """
        df = df.copy()
        atr = self._atr(df, self.atr_period)
        
        labels = pd.Series(index=df.index, dtype=float)
        labels[:] = -1  # Default: skip
        
        for i in range(len(df) - self.max_bars):
            if pd.isna(atr.iloc[i]):
                continue
                
            entry = df['close'].iloc[i]
            stop_loss = entry - (atr.iloc[i] * self.risk_multiple) if direction == 'long' else entry + (atr.iloc[i] * self.risk_multiple)
            take_profit = entry + (atr.iloc[i] * self.reward_multiple) if direction == 'long' else entry - (atr.iloc[i] * self.reward_multiple)
            
            future_bars = df.iloc[i+1:i+1+self.max_bars]
            
            # Check if stop or target hit first
            if direction == 'long':
                stop_hit = (future_bars['low'] <= stop_loss).any()
                target_hit = (future_bars['high'] >= take_profit).any()
            else:
                stop_hit = (future_bars['high'] >= stop_loss).any()
                target_hit = (future_bars['low'] <= take_profit).any()
            
            if target_hit and not stop_hit:
                labels.iloc[i] = 1
            elif stop_hit and not target_hit:
                labels.iloc[i] = 0
            elif stop_hit and target_hit:
                # Ambiguous - determine which came first
                labels.iloc[i] = self._determine_first_hit(future_bars, stop_loss, take_profit, direction)
            else:
                # Neither hit - timeout, treat as loss
                labels.iloc[i] = 0
        
        return labels
    
    def _determine_first_hit(self, future_bars: pd.DataFrame, stop: float, target: float, direction: str) -> int:
        """Determine if stop or target was hit first."""
        for _, bar in future_bars.iterrows():
            if direction == 'long':
                if bar['low'] <= stop:
                    return 0
                if bar['high'] >= target:
                    return 1
            else:
                if bar['high'] >= stop:
                    return 0
                if bar['low'] <= target:
                    return 1
        return 0  # Default to loss if ambiguous
    
    def _atr(self, df: pd.DataFrame, period: int) -> pd.Series:
        high_low = df['high'] - df['low']
        high_close = abs(df['high'] - df['close'].shift())
        low_close = abs(df['low'] - df['close'].shift())
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(window=period).mean()

In [12]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, f1_score
import joblib
from datetime import datetime, timedelta

class XGBoostForexFilter:
    """
    Walk-forward training to prevent look-ahead bias and overfitting.
    """
    
    def __init__(self,
                 train_window: int = 5000,    # Bars for training
                 test_window: int = 500,      # Bars for testing
                 step_size: int = 500,        # Step forward amount
                 threshold: float = 0.6):     # Probability threshold for trade
        self.train_window = train_window
        self.test_window = test_window
        self.step_size = step_size
        self.threshold = threshold
        self.model = None
        self.feature_cols = None
        self.scaler = None  # Optional: add StandardScaler if needed
    
    def prepare_data(self, df: pd.DataFrame, direction: str = 'long') -> pd.DataFrame:
        """Full pipeline: features + labels."""
        # Generate features
        fe = ForexFeatureEngineer()
        df = fe.compute_features(df)
        
        # Generate labels
        lg = LabelGenerator()
        df['target'] = lg.generate_labels(df, direction)
        
        # Drop rows with NaN in features or target
        self.feature_cols = fe.get_feature_columns(df)
        df = df.dropna(subset=self.feature_cols + ['target'])
        
        # Only keep rows where target is 0 or 1 (drop -1 uncertain)
        df = df[df['target'].isin([0, 1])]
        
        return df
    
    def walk_forward_train(self, df: pd.DataFrame) -> Dict:
        """
        Train multiple models on rolling windows.
        Returns performance metrics per fold.
        """
        df = df.reset_index(drop=True)
        n_samples = len(df)
        
        results = []
        model_idx = 0
        
        for start in range(0, n_samples - self.train_window - self.test_window, self.step_size):
            train_start = start
            train_end = start + self.train_window
            test_start = train_end
            test_end = test_start + self.test_window
            
            # Split
            train_df = df.iloc[train_start:train_end]
            test_df = df.iloc[test_start:test_end]
            
            X_train = train_df[self.feature_cols]
            y_train = train_df['target']
            X_test = test_df[self.feature_cols]
            y_test = test_df['target']
            
            # Train
            model = xgb.XGBClassifier(
                n_estimators=200,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
                eval_metric='logloss',
                random_state=42
            )
            
            model.fit(
                X_train, y_train,
                eval_set=[(X_test, y_test)],
                early_stopping_rounds=20,
                verbose=False
            )
            
            # Evaluate
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
            
            # Only trade when probability > threshold
            trades = y_proba >= self.threshold
            if trades.sum() > 0:
                trade_accuracy = (y_test[trades] == 1).mean()
            else:
                trade_accuracy = 0
            
            results.append({
                'fold': model_idx,
                'train_start': train_start,
                'test_start': test_start,
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'f1': f1_score(y_test, y_pred, zero_division=0),
                'trade_accuracy': trade_accuracy,
                'trade_rate': trades.mean(),
                'n_trades': trades.sum()
            })
            
            # Save best model (you can define "best" differently)
            if model_idx == 0 or trade_accuracy > max(r['trade_accuracy'] for r in results[:-1]):
                self.model = model
                joblib.dump(model, f'model_fold_{model_idx}.pkl')
            
            model_idx += 1
        
        return pd.DataFrame(results)
    
    def predict(self, features: pd.DataFrame) -> Tuple[int, float]:
        """
        Returns: (trade_signal, probability)
        trade_signal: 1 = take trade, 0 = no trade
        """
        if self.model is None:
            raise ValueError("Model not trained")
        
        proba = self.model.predict_proba(features)[:, 1][0]
        signal = 1 if proba >= self.threshold else 0
        return signal, proba
    
    def feature_importance(self) -> pd.Series:
        """Return feature importance from trained model."""
        if self.model is None:
            raise ValueError("Model not trained")
        return pd.Series(
            self.model.feature_importances_,
            index=self.feature_cols
        ).sort_values(ascending=False)

In [13]:
import oandapyV20
from oandapyV20.endpoints.instruments import InstrumentsCandles
import config  # Your OANDA credentials

class LiveXGBFilter:
    """
    Live inference wrapper for XGBoost filter.
    """
    
    def __init__(self, model_path: str, instrument: str, granularity: str = 'H1'):
        self.model = joblib.load(model_path)
        self.feature_engineer = ForexFeatureEngineer()
        self.instrument = instrument
        self.granularity = granularity
        self.client = oandapyV20.API(access_token=config.OANDA_TOKEN)
        
        # Buffer for historical data
        self.price_buffer = pd.DataFrame()
        self.min_bars = 100  # Minimum bars needed for features
    
    def fetch_latest_candles(self, count: int = 500) -> pd.DataFrame:
        """Fetch recent candles from OANDA."""
        params = {
            "count": count,
            "granularity": self.granularity,
            "price": "M"  # Midpoint
        }
        
        r = InstrumentsCandles(instrument=self.instrument, params=params)
        self.client.request(r)
        
        candles = r.response['candles']
        data = []
        for c in candles:
            if not c['complete']:
                continue
            data.append({
                'time': pd.to_datetime(c['time']),
                'open': float(c['mid']['o']),
                'high': float(c['mid']['h']),
                'low': float(c['mid']['l']),
                'close': float(c['mid']['c']),
                'volume': int(c['volume'])
            })
        
        df = pd.DataFrame(data).set_index('time')
        return df
    
    def update_and_predict(self) -> dict:
        """
        Fetch latest data, compute features, return prediction.
        Returns dict with signal details.
        """
        # Fetch data
        df = self.fetch_latest_candles(count=500)
        
        # Compute features
        df = self.feature_engineer.compute_features(df)
        
        # Get latest complete bar's features
        latest = df.iloc[-2]  # -2 because -1 might be incomplete
        feature_cols = self.feature_engineer.get_feature_columns(df)
        
        # Ensure no NaN
        if latest[feature_cols].isna().any():
            return {'signal': 0, 'error': 'Insufficient data for features'}
        
        X = latest[feature_cols].values.reshape(1, -1)
        proba = self.model.predict_proba(X)[:, 1][0]
        
        return {
            'signal': 1 if proba >= 0.6 else 0,
            'probability': float(proba),
            'timestamp': latest.name.isoformat(),
            'close': float(latest['close']),
            'features': {k: float(v) for k, v in latest[feature_cols].items()}
        }

In [14]:
# === TRAINING ===
# Load your historical data (from your database)
df = pd.read_sql("SELECT * FROM eurusd_h1 ORDER BY time", engine)

# Initialize and train
filter_model = XGBoostForexFilter(
    train_window=3000,
    test_window=500,
    step_size=500,
    threshold=0.6
)

# Prepare data
prepared_df = filter_model.prepare_data(df, direction='long')

# Walk-forward training
results = filter_model.walk_forward_train(prepared_df)
print(results[['fold', 'trade_accuracy', 'trade_rate', 'n_trades']])

# Check feature importance
print(filter_model.feature_importance().head(10))

# Save final model
joblib.dump(filter_model.model, 'xgboost_filter_final.pkl')

# === LIVE ===
live = LiveXGBFilter('xgboost_filter_final.pkl', 'EUR_USD', 'H1')
signal = live.update_and_predict()
print(signal)
# {'signal': 1, 'probability': 0.73, 'timestamp': '...', 'close': 1.0850, ...}

NameError: name 'engine' is not defined